# Level 2 — Spatial Context: Mapping the Glioma Microenvironment

## CAJAL "Neuromics 2026" — Computational Mini-Project C10 (Level 2)

**Estimated time:** ~2 days

**Learning objectives**
- Load, QC, and explore Visium spatial transcriptomics data
- Get a first ("naive") spatial domain map directly from spot expression, before any deconvolution
- See the malignant cell-state axis directly in space, even before deconvolving spots
- Map your Level 1 reference onto tissue with **cell2location**
- Identify spatial niches (tissue domains) from the deconvolved cell-state map
- Quantify spatial organization: neighborhood enrichment, a proximity network, and spatial intermixing
- Compare your own results to the published figures (the paper is revealed in this notebook)

**Dataset:** Visium spatial transcriptomics from the **same two donors** as Level 1 — `AT10`
(primary, full feature set) and `AT14` (optional secondary section). Each spot covers
multiple cells (~1-10), so spot expression is a *mixture*, unlike Level 1's single nuclei.

> **The paper behind this project — read it first.** You'll be reproducing analyses from
> **de Jong, Memi, Gracia, Lazareva *et al.*, "A spatiotemporal cancer cell trajectory underlies
> glioblastoma heterogeneity," bioRxiv 2025.05.13.653495** (companion website
> **[gbmspace.org](https://www.gbmspace.org/)**). **Before you start, skim the paper — the abstract
> and Figures 1–3 — and browse gbmspace.org** to get the big picture: 12 IDH-wildtype glioblastomas
> profiled with snRNA-seq + Visium (+ Xenium), showing that malignant cells occupy a *continuous*
> developmental → gliosis → hypoxia trajectory that maps onto spatial **zonation** in the tumour.
> You'll recognise the cell-type and cell-state language from Level 1 — that continuity is the point.
> Section 9 revisits the paper's key findings in detail so you can compare them against your own results.

---

> ## Before you start — metadata your Level 1 output must contain
>
> Level 2 loads **your annotated snRNA-seq reference from Level 1** and maps it onto the spatial data
> with cell2location. Your Level 1 run may have produced slightly different column names or label
> values, so **check your annotated AnnData has all of the following before proceeding** — the
> reference-setup step (Section 5) will fail without them:
>
> | Where | Key | What it is |
> |---|---|---|
> | `.layers["counts"]` | raw integer counts | cell2location's model needs **raw** counts, not log-normalized `.X` |
> | `.obs["donor_id"]` | donor / batch | used as the cell2location `batch_key` |
> | `.obs["cell_type"]` | coarse cell-type labels | TME + lineage annotation (Level 1 §6) |
> | `.obs["cell_status_derived"]` | malignant vs TME, with the value `"Malignant"` | splits malignant from TME (Level 1 §7) |
> | `.obs["malignant_state"]` | malignant cell-state labels | the malignant states (Level 1 §8); only needs values on malignant nuclei |
>
> From these, Level 2 builds the reference label `cell_state_for_c2l` = `malignant_state` for
> malignant nuclei, else `cell_type`. **Quick self-check** (run once your reference is loaded):
>
> ```python
> adata_ref = sc.read_h5ad(ref_path)
> assert "counts" in adata_ref.layers, "no raw-counts layer"
> for col in ["donor_id", "cell_type", "cell_status_derived", "malignant_state"]:
>     assert col in adata_ref.obs, f"missing .obs['{col}']"
> assert (adata_ref.obs["cell_status_derived"] == "Malignant").any(), "no cells flagged 'Malignant'"
> print("OK — your reference has the metadata Level 2 needs")
> ```
>
> If a name differs (e.g. you called it `malignant_class` instead of `malignant_state`), either
> rename it to match, or adjust the reference-setup cell in Section 5.

## 0. Setup

In [ ]:
# === PROVIDED - project paths. Run this first; nothing to edit. ===
# Every path below is derived from the repo root, so this notebook runs from any
# checkout location (cluster, laptop, Colab) without editing a single path.
# If you keep the datasets somewhere else, set the C10_ROOT environment variable.
import os
import sys
from pathlib import Path


def _find_project_root(start):
    for cand in [start, *start.parents]:
        if (cand / "src" / "gbmspace_utils").is_dir():
            return cand
    raise RuntimeError(
        f"Could not find the gbm-space-c10 repo root at or above {start}. "
        "Start Jupyter from inside the repo, or set C10_ROOT to the repo path."
    )


PROJECT_ROOT = (
    Path(os.environ["C10_ROOT"]).expanduser().resolve()
    if os.environ.get("C10_ROOT")
    else _find_project_root(Path.cwd().resolve())
)
NOTEBOOK_DIR = Path.cwd().resolve()   # figures are written here, next to the notebook
DATA_DIR = PROJECT_ROOT / "data"
PRECOMP_DIR = PROJECT_ROOT / "precomputed"
REFERENCE_DIR = PROJECT_ROOT / "reference"

sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
# === PROVIDED — setup (imports + project helpers + paths). Run this first; nothing to edit. ===
# (If you cloned the repo elsewhere, update the sys.path.insert line to point at your repo's src/ folder.)
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from gbmspace_utils.analysis import (
    MALIGNANT_AXIS_MARKERS, MAJOR_CLASS_OF, ZONATION_PANEL, score_axis,
    assign_dominant_state, spatial_proximity_network,
)
from gbmspace_utils.plotting import plot_gene_on_tissue, plot_spatial_categories

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, frameon=False, figsize=(5, 4))
%matplotlib inline

print("scanpy", sc.__version__, "| squidpy", sq.__version__)

## 1. Load and explore the spatial data

**TASK 1.1:** Load the AT10 Visium section and inspect the object — note how it differs from Level 1's AnnData (spatial coordinates, a tissue image, no per-nucleus QC metrics yet).

> **Data structure — is this a *SpatialData* object?** No. Your Visium section is a plain
> **`AnnData`** — the same object type as Level 1 — just with spatial information attached. There is
> **no separate SpatialData object** in this course. What differs from Level 1:
>
> | Part | Level 1 (snRNA-seq) | Level 2 (Visium) |
> |---|---|---|
> | one row of `.obs` = | one **nucleus** | one **spot** (pools ~1–10 cells) |
> | `.X` | expression per nucleus | expression per spot (**raw counts** here) |
> | `.var` | genes | genes |
> | `.obsm["spatial"]` | — | **(x, y) pixel coordinates** of each spot on the slide |
> | `.uns["spatial"]` | — | the **H&E tissue image** + scale factors (for plotting on tissue) |
> | `.obs` extras | QC, labels | `array_row` / `array_col` (spot grid), `in_tissue` |
>
> So everything you know about AnnData still applies (`sc.pp.*`, `sc.tl.*`, `.obs`, `.layers`). The
> only new idea is that each observation has a **location** — which lets you plot values *on the
> tissue* (with `squidpy.pl.spatial_scatter`, or by scattering `.obsm["spatial"]`). *(Aside: the
> newer `spatialdata` library offers a richer multi-modal container, but this course stays with
> AnnData — it's all you need here.)*

In [ ]:
# --- TASK 1.1: load the AT10 Visium section ---
# WHAT & WHY: read the spatial AnnData and get oriented. Note it's the SAME AnnData type as Level 1,
#             plus .obsm['spatial'] (spot coordinates) and .uns['spatial'] (the tissue image).
# HOW: VISIUM = "<AT10 Visium .h5ad>"
#      adata = sc.read_h5ad(VISIUM); print(adata); print(list(adata.obsm), list(adata.uns))
# DOCS: https://anndata.readthedocs.io/en/stable/generated/anndata.read_h5ad.html

# ---- provided — read it and run it (mostly plotting/formatting) ----
VISIUM = DATA_DIR / "visium/level2_prepared/AT10-BRA-5-FO-1_2_student.h5ad"
adata = sc.read_h5ad(VISIUM)   # load the AnnData object from disk
print(adata)
print(f"\n{adata.n_obs} spots x {adata.n_vars} genes")
print(f".obsm: {list(adata.obsm.keys())}  |  .uns: {list(adata.uns.keys())}")
lib_id = list(adata.uns["spatial"].keys())[0]
print(f"Library: {lib_id}, images: {list(adata.uns['spatial'][lib_id]['images'].keys())}")

In [ ]:
# --- Look at the tissue ---
# WHAT & WHY: view the H&E image with the spot grid on top, so you can see the tissue you're analysing.
# HOW: sq.pl.spatial_scatter(adata, color=None, size=1.3)   # color=None -> just the image + spots
# DOCS: https://squidpy.readthedocs.io/en/stable/api/squidpy.pl.spatial_scatter.html

# ---- provided — read it and run it (mostly plotting/formatting) ----
fig, ax = plt.subplots(figsize=(6, 6))   # set up the figure and its panel(s)
sq.pl.spatial_scatter(adata, color=None, ax=ax, size=1.3)   # draw the spots at their tissue positions (colour set by color=)
ax.set_title(f"{lib_id} — H&E + spot grid ({adata.n_obs} spots)")   # give this panel a title

**QUESTION:** Each Visium spot is ~55 µm in diameter. Given typical nucleus/cell sizes, roughly how many cells might a single spot cover? What does that imply about interpreting any single spot's gene expression?

## 2. Spatial quality control

Same idea as Level 1 — total counts, genes detected, %mito — but spots, not nuclei, and no doublet score (a "doublet" concept doesn't apply the same way to multi-cell spots).

**TASK 2.1:** Compute QC metrics and look at their distributions.

In [ ]:
# --- TASK 2.1: compute spatial QC metrics ---
# WHAT & WHY: flag mitochondrial genes and compute per-SPOT QC (total counts, genes detected, %mito).
# HOW: adata.var['mt'] = adata.var_names.str.startswith('MT-')
#      sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, inplace=True)
# DOCS: https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.calculate_qc_metrics.html

# ---- the plotting is provided — complete the steps (single-cell pipeline you know from Level 1) ----
adata.var["mt"] = adata.var_names.str.startswith("MT-")   # flag mitochondrial genes
...   # your code here — complete this step (you did this in Level 1)
print(adata.obs[["total_counts", "n_genes_by_counts", "pct_counts_mt"]].describe().round(1))

In [ ]:
# --- Look at the QC distributions ---
# WHAT & WHY: see the shape of the distributions before choosing thresholds (see the HINT below).
# HOW: histograms of adata.obs['total_counts'], ['n_genes_by_counts'], ['pct_counts_mt'] (plt.hist, bins=60).
# DOCS: https://scanpy.readthedocs.io/en/stable/generated/scanpy.pl.violin.html   (sc.pl.violin is an alternative)

# ---- provided — read it and run it (mostly plotting/formatting) ----
fig, axes = plt.subplots(1, 3, figsize=(13, 4))   # set up the figure and its panel(s)
axes[0].hist(adata.obs["total_counts"], bins=60)   # histogram of these values
axes[0].set_title("Total counts")   # give this panel a title
axes[1].hist(adata.obs["n_genes_by_counts"], bins=60)   # histogram of these values
axes[1].set_title("Genes detected")   # give this panel a title
axes[2].hist(adata.obs["pct_counts_mt"], bins=60)   # histogram of these values
axes[2].set_title("% mitochondrial")   # give this panel a title
plt.tight_layout()   # adjust spacing so panels don't overlap
plt.show()   # display the figure

**HINT:** Visium spots are much "deeper" than single nuclei (each spot pools several cells), so don't reuse Level 1's per-nucleus thresholds verbatim — look at *these* distributions. A light touch is usually right: drop near-empty spots (very low counts/genes), keep almost everything else.

**TASK 2.2:** Apply QC and a minimum-cells gene filter. Report spots/genes remaining.

In [ ]:
# --- TASK 2.2: apply QC (light touch for Visium) ---
# WHAT & WHY: drop only near-empty spots, and genes seen in very few spots. Keep almost everything.
# HOW: adata = adata[(adata.obs['total_counts'] >= 500) & (adata.obs['n_genes_by_counts'] >= 250)].copy()
#      sc.pp.filter_genes(adata, min_cells=3);  print spots/genes remaining
# DOCS: https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.filter_genes.html

# ---- the plotting is provided — complete the steps (single-cell pipeline you know from Level 1) ----
n0 = adata.n_obs
adata = adata[(adata.obs["total_counts"] >= 500) & (adata.obs["n_genes_by_counts"] >= 250)].copy()
...   # your code here — complete this step (you did this in Level 1)
print(f"Spots: {n0} -> {adata.n_obs}")
print(f"Genes (min_cells=3): {adata.n_vars}")

**CHECKPOINT:** This section should remove very few spots — on the order of **1-2% of spots** (this dataset is high quality; expect roughly **3,900-3,950 spots** remaining out of ~4,000, and somewhere around **24,000-25,000 genes** after the gene filter). If you lost a large fraction of spots, your thresholds are too strict for Visium-scale counts.

## 3. Normalization and a *naive* spatial domain map

Before any deconvolution, let's see what plain clustering of spot expression gives us — a "naive" map of spatial domains, mixing whatever cell types happen to co-occur in each spot.

**TASK 3.1:** Normalize, log-transform, select HVGs, and run PCA.

In [ ]:
# --- TASK 3.1: normalize, log, HVGs, PCA ---
# WHAT & WHY: stash raw counts (needed later), normalize+log so spots are comparable, pick the most
#             variable genes, then PCA to compress before clustering.
# HOW: adata.layers['counts'] = adata.X.copy(); sc.pp.normalize_total(adata, target_sum=1e4); sc.pp.log1p(adata); adata.raw = adata
#      sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat_v3', layer='counts')
#      adata_hvg = adata[:, adata.var['highly_variable']].copy(); sc.pp.scale(adata_hvg, max_value=10); sc.tl.pca(adata_hvg, n_comps=30)
#      (copy adata_hvg.obsm['X_pca'] back onto adata.obsm['X_pca'])
# DOCS: https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.normalize_total.html  |  https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.highly_variable_genes.html  |  https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.pca.html

# ---- the plotting is provided — complete the steps (single-cell pipeline you know from Level 1) ----
adata.layers["counts"] = adata.X.copy()
...   # your code here — complete this step (you did this in Level 1)
adata.raw = adata
...   # your code here — complete this step (you did this in Level 1)
adata_hvg = adata[:, adata.var["highly_variable"]].copy()
...   # your code here — complete this step (you did this in Level 1)
adata.obsm["X_pca"] = adata_hvg.obsm["X_pca"]
adata.uns["pca"] = adata_hvg.uns["pca"]
sc.pl.pca_variance_ratio(adata, n_pcs=30, log=True)   # scanpy plotting call

**TASK 3.2:** Build the neighbor graph, UMAP, and cluster at a couple of resolutions. Pick one.

In [ ]:
# --- TASK 3.2: neighbours, UMAP, Leiden clustering ---
# WHAT & WHY: build the graph + UMAP, then cluster spots into 'naive' domains at a couple of resolutions.
# HOW: sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30); sc.tl.umap(adata)
#      for res in (0.5, 1.0): sc.tl.leiden(adata, resolution=res, key_added=f'leiden_r{res}', flavor='igraph', n_iterations=2)
#      adata.obs['leiden'] = adata.obs['leiden_r1.0']
# DOCS: https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.neighbors.html  |  https://scanpy.readthedocs.io/en/stable/generated/scanpy.tl.umap.html  |  https://scanpy.readthedocs.io/en/stable/generated/scanpy.tl.leiden.html

# ---- the plotting is provided — complete the steps (single-cell pipeline you know from Level 1) ----
...   # your code here — complete this step (you did this in Level 1)
for res in [0.5, 1.0]:
    ...   # your code here — complete this step (you did this in Level 1)
    print(f"resolution {res}: {adata.obs[f'leiden_r{res}'].nunique()} naive domains")
adata.obs["leiden"] = adata.obs["leiden_r1.0"]

At resolution 0.5 you should see roughly **10-12** domains; at 1.0, roughly **16-20**. We carry resolution 1.0 forward as `leiden` — finer domains are easier to relate to the cell-state axis later, but feel free to use 0.5 instead.

**TASK 3.3:** Plot the naive domains both on the tissue and on UMAP, side by side.

In [ ]:
# --- TASK 3.3: plot the naive domains on tissue AND on UMAP ---
# WHAT & WHY: compare the same clustering in space vs in embedding space -- are domains contiguous patches?
# HOW: fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
#      sq.pl.spatial_scatter(adata, color='leiden', ax=axes[0], size=1.3)
#      sc.pl.umap(adata, color='leiden', ax=axes[1], show=False)
# DOCS: https://squidpy.readthedocs.io/en/stable/api/squidpy.pl.spatial_scatter.html  |  https://scanpy.readthedocs.io/en/stable/generated/scanpy.pl.umap.html

# ---- provided — read it and run it (mostly plotting/formatting) ----
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))   # set up the figure and its panel(s)
sq.pl.spatial_scatter(adata, color="leiden", ax=axes[0], size=1.3, legend_fontsize=6)   # draw the spots at their tissue positions (colour set by color=)
axes[0].set_title("Naive spatial domains (tissue)")   # give this panel a title
sc.pl.umap(adata, color="leiden", ax=axes[1], show=False, title="Naive spatial domains (UMAP)")   # draw the UMAP embedding (colour set by color=)
plt.tight_layout()   # adjust spacing so panels don't overlap
plt.show()   # display the figure

**QUESTION:** Do the naive domains form spatially coherent regions (contiguous patches of one color), or are they speckled/scattered across the tissue? What would each pattern imply about whether expression-based clustering alone is picking up real tissue architecture?

## 4. The malignant cell-state axis, in space (before deconvolution)

Level 1's marker-gene scoring works the same way here — `score_genes` doesn't care whether
an observation is a nucleus or a multi-cell spot. The catch: a spot's score is a *blend* of
whatever cell states happen to be in that spot, not a clean call.

**TASK 4.1:** Score every spot against the Level 1 malignant-state marker sets, using the shared `score_axis()` helper.

In [ ]:
# --- TASK 4.1: score the malignant-state axis per spot ---
# WHAT & WHY: score every spot for the Level 1 marker sets. A spot's score is a BLEND of its cells --
#             that's the limitation cell2location will fix later.
# HOW: state_scores = score_axis(adata, MALIGNANT_AXIS_MARKERS, use_raw=True)   # helper from gbmspace_utils
#      for col in state_scores.columns: adata.obs[f'score_{col}'] = state_scores[col].values
# DOCS: https://scanpy.readthedocs.io/en/stable/generated/scanpy.tl.score_genes.html   (score_axis wraps sc.tl.score_genes; see src/gbmspace_utils/analysis.py)

# ---- the plotting is provided — complete the steps (single-cell pipeline you know from Level 1) ----
...   # your code here — complete this step (you did this in Level 1)
for col in state_scores.columns:
    adata.obs[f"score_{col}"] = state_scores[col].values
print(adata.obs.groupby("leiden")[[f"score_{c}" for c in state_scores.columns]].mean().round(3))   # group the rows to aggregate per group

**TASK 4.2:** Plot the paper's minimal 4-gene spatial zonation panel (`ZONATION_PANEL`: dev-like → gliosis → hypoxia) directly on tissue.

In [ ]:
# --- TASK 4.2: plot the 4-gene zonation panel on tissue ---
# WHAT & WHY: look for a spatial gradient across AQP4 -> ABCC3 -> AKAP12 -> HILPDA (dev-like -> gliosis -> hypoxia).
# HOW: for gene in ZONATION_PANEL (that are in adata.raw.var_names): scatter adata.obsm['spatial'][:,0/1]
#      coloured by that gene's expression; or sq.pl.spatial_scatter(adata, color=gene). One subplot per gene.
# DOCS: https://squidpy.readthedocs.io/en/stable/api/squidpy.pl.spatial_scatter.html

# ---- provided — read it and run it (mostly plotting/formatting) ----
present = [g for g in ZONATION_PANEL if g in adata.raw.var_names]
fig2, axes2 = plt.subplots(1, len(present), figsize=(4.2 * len(present), 4.2))   # set up the figure and its panel(s)
for ax, gene in zip(axes2, present):
    expr = np.asarray(adata[:, gene].X.todense()).flatten()
    sca = ax.scatter(adata.obsm["spatial"][:, 0], adata.obsm["spatial"][:, 1], c=expr, cmap="Reds", s=8,   # scatter each point at its (x, y); colour = the c= values
                      vmax=np.percentile(expr[expr > 0], 95) if (expr > 0).any() else None)
    ax.invert_yaxis()   # flip the y-axis so it matches the tissue-image orientation
    ax.set_aspect("equal")   # keep x and y on the same scale so the tissue isn't distorted
    ax.set_title(gene)   # give this panel a title
    ax.axis("off")   # hide the axis frame and ticks
    fig2.colorbar(sca, ax=ax, shrink=0.7)   # add a colour-scale legend
plt.tight_layout()   # adjust spacing so panels don't overlap
plt.show()   # display the figure

**QUESTION:** Do you see any spatial gradient across the four zonation genes — e.g. do `AQP4`-high and `HILPDA`-high regions occupy *different, non-overlapping* areas of the tissue? Compare this to the per-cluster axis scores above. This blended, spot-level picture is exactly the limitation **cell2location** is designed to address — keep this figure in mind for comparison once you've deconvolved.

---

## 5. Mapping single cells onto space with cell2location

So far every spot has been treated as one observation, even though it's really a mixture.
**cell2location** uses your Level 1 reference (cell types learned from single nuclei) to
estimate *how many cells of each type* are in every spot — turning "this spot's expression
looks bulk hypoxic" into "this spot is ~60% Hypoxic-state malignant cells + ~20% Macrophage + ...".

**TASK 5.1:** Load your saved, annotated Level 1 reference, and build the label cell2location
will actually deconvolve.

> **How cell2location works (plain-language).** Each Visium spot's expression is a *sum* of the
> cells sitting inside it. cell2location is a Bayesian model that "un-mixes" that sum, in two stages:
>
> 1. **Reference signature model** — a negative-binomial regression on your Level 1 single-nucleus
>    counts. It learns, for every gene, its typical expression level in each reference cell type/state:
>    a genes × cell-types **signature** matrix (the `inf_aver` table).
> 2. **Spatial mapping model** — for every spot it infers *how many cells of each reference type* are
>    present, so that (signatures × estimated cell-counts) best reproduces that spot's observed counts.
>    It shares information across spots and corrects for each spot's technical sensitivity. Fitting is
>    done by **variational inference** (gradient descent on the model's ELBO) — that's why it's slow on
>    CPU and much faster on a GPU (so we load a precomputed result here).
>
> **Output:** a spots × cell-types **abundance** matrix — the estimated number of each cell type in
> each spot. Being Bayesian, it also gives uncertainty; we use the conservative 5% quantile (`q05`).
> Two knobs worth knowing: `N_cells_per_location` (expected cells per spot, ~30 for Visium) and
> `detection_alpha` (how much per-spot sensitivity may vary). It needs **raw counts** (the NB model is
> defined on integers) — that's why the reference is set up with `layer="counts"`.

In [ ]:
# --- TASK 5.1: load your Level 1 annotated reference ---
# WHAT & WHY: cell2location needs your single-nucleus reference (cell types learned in Level 1).
# HOW: ref_path = "<annotated Level 1 reference .h5ad>"; adata_ref = sc.read_h5ad(ref_path)
#      Then run the self-check from the yellow box at the TOP of this notebook to confirm it has the
#      counts layer + the required .obs columns before continuing.
# DOCS: https://anndata.readthedocs.io/en/stable/generated/anndata.read_h5ad.html

# ---- provided — read it and run it (mostly plotting/formatting) ----
ref_path = DATA_DIR / "processed/gbm_l1_snrna_AT10_AT14_annotated.h5ad"
adata_ref = sc.read_h5ad(ref_path)   # load the AnnData object from disk
print(f"Reference: {adata_ref.n_obs} nuclei, cell types: {adata_ref.obs['cell_type'].nunique()}")
print(adata_ref.obs['cell_type'].value_counts())

**HINT — which label should cell2location deconvolve?** `cell_type` (Section 6's CellTypist
majority call per cluster) is a reasonable broad label for TME cells, but for malignant cells it's
a region-mimic label (e.g. "Hypothalamus glioblast" vs "Striatum glioblast") — the same malignant
population matched to whichever normal-brain-region profile CellTypist happened to land on, not a
real biological distinction (their average expression profiles are correlated >0.9). Per the
paper's own Methods, cell2location was run on genuine **malignant cell-state clusters**, not
region-mimic labels. Build a combined label: `malignant_state` (Section 8's 9 marker-defined axis
states) for malignant cells, `cell_type` for TME cells.

Some non-malignant clusters carry a malignant-mimic CellTypist label too (their CNV signal
didn't clear the malignant threshold, so they're correctly TME — but CellTypist still gave them a
"glioblast"/"OPC"-sounding name). Don't just drop these: check their actual marker-gene expression
first. A quick oligodendrocyte marker score (`MBP, PLP1, MOG, MOBP, ST18`) reveals most of them
*are* real oligodendrocytes (mean score ~120-155 vs. ~1-6 in true TME/malignant cells) — CellTypist's
`Developing_Human_Brain` model has no clean adult-oligodendrocyte category, so it matched them to
the closest thing it had (developing OPC/neural-crest programmes). Relabel those properly instead
of throwing away a real, abundant TME population; only drop the few small clusters that show
*neither* a malignant-state signature *nor* oligodendrocyte markers (truly ambiguous, ~2% of cells).

**TASK 5.1b:** Build `cell_state_for_c2l` and use it as the reference label.

In [ ]:
# === PROVIDED — data-cleaning (you don't need to write this). ===
# Relabels CellTypist 'mimic' TME cells that are really oligodendrocytes (MBP/PLP1/MOG/MOBP/ST18),
# drops the few truly-ambiguous cells, then builds `cell_state_for_c2l` (the label cell2location
# deconvolves). See the HINT above for the full reasoning.
MALIGNANT_MIMIC_KEYWORDS = ("glioblast", "radial glia", "opc", "neural crest", "neuroblast")
OLIGODENDROCYTE_MARKERS = ["MBP", "PLP1", "MOG", "MOBP", "ST18"]

is_malignant = (adata_ref.obs["cell_status_derived"] == "Malignant").to_numpy()
is_mimic_but_tme = (~is_malignant) & adata_ref.obs["cell_type"].str.lower().str.contains(
    "|".join(MALIGNANT_MIMIC_KEYWORDS)).to_numpy()

present_markers = [g for g in OLIGODENDROCYTE_MARKERS if g in adata_ref.var_names]
counts = adata_ref.layers["counts"]
libsize = np.asarray(counts.sum(axis=1)).flatten()
marker_sum = np.asarray(counts[:, [adata_ref.var_names.get_loc(g) for g in present_markers]].sum(axis=1)).flatten()
oligo_score = marker_sum / np.maximum(libsize, 1) * 1e4

is_real_oligo = is_mimic_but_tme & (oligo_score > 20)  # >>10x the ~1-6 baseline in true TME/malignant cells
is_truly_ambiguous = is_mimic_but_tme & ~is_real_oligo
print(f"Of {int(is_mimic_but_tme.sum())} mimic-labelled-but-TME nuclei: "
      f"{int(is_real_oligo.sum())} are real oligodendrocytes (relabelled), "
      f"{int(is_truly_ambiguous.sum())} stay ambiguous (dropped)")

adata_ref = adata_ref[~is_truly_ambiguous].copy()
is_malignant = (adata_ref.obs["cell_status_derived"] == "Malignant").to_numpy()
is_real_oligo = is_real_oligo[~is_truly_ambiguous]

cell_type_relabelled = adata_ref.obs["cell_type"].astype(str).to_numpy()
cell_type_relabelled[is_real_oligo] = "Oligodendrocyte"
adata_ref.obs["cell_state_for_c2l"] = np.where(
    is_malignant, adata_ref.obs["malignant_state"].astype(str), cell_type_relabelled)
print("\nReference labels for cell2location:")
print(adata_ref.obs["cell_state_for_c2l"].value_counts())

**HINT — runtime.** cell2location is two models: a reference **signature** model (NB regression on your Level 1 single-cell counts) and a **spatial mapping** model (maps that signature onto spots). Both are slow on CPU. We benchmarked this on the actual data: training on all shared genes costs **~170s/epoch**; the standard cell2location gene filter (`cell2location.utils.filtering.filter_genes`, defaults) cuts that to **~15,900 genes** and **~72s/epoch** (reference) / **~3.9s/epoch** (spatial mapping, this Visium section's spot count). Paper-faithful epoch counts (ref 400 / mapping 6000) would take **~8 hours total on CPU** — only realistic on GPU. Set the mode below accordingly.

**TASK 5.2:** Set the compute mode, filter genes, and train the reference signature model.

**HINT:** cell2location's reference model assumes a negative-binomial (GammaPoisson) likelihood over **raw integer counts** — but by this point your Level 1 reference's `.X` is log-normalized (from Level 1 Section 3). Point `setup_anndata` at the raw-counts layer explicitly (`layer="counts"`), or training will crash with a cryptic "value... not within the support of GammaPoisson" error the first time it tries to evaluate a likelihood on fractional log-values.

In [ ]:
# [KEEP-IN-STUDENT]
import os
C2L_MODE = "FULL"   # "DEMO" / "FAST" (CPU-runnable) / "FULL" (paper-exact; needs a GPU to train)
REF_EPOCHS = {"DEMO": 5, "FAST": 20, "FULL": 400}[C2L_MODE]
MAP_EPOCHS = {"DEMO": 20, "FAST": 300, "FULL": 6000}[C2L_MODE]
print(f"Mode={C2L_MODE}: reference {REF_EPOCHS} epochs, mapping {MAP_EPOCHS} epochs")

# cell2location training (reference signature + per-section spatial mapping) is the GPU step.
# It was run for you at FULL and the outputs saved in precomputed/; the cells below LOAD those and
# skip training. To train it yourself, set TRAIN_C2L = True (FULL needs a GPU; FAST runs on CPU).
TRAIN_C2L = False
# PRECOMP_DIR comes from the project-paths cell at the top.
CKPT_DIR = PROJECT_ROOT / "scratch_build/checkpoints"   # instructor training cache (fallback)
os.makedirs(CKPT_DIR, exist_ok=True)

In [ ]:
# [KEEP-IN-STUDENT]
from cell2location.utils.filtering import filter_genes
from cell2location.models import RegressionModel, Cell2location

shared = sorted(set(adata_ref.var_names) & set(adata.var_names))
ref = adata_ref[:, shared].copy()
vis = adata.copy()[:, shared].copy()

selected = filter_genes(ref, cell_count_cutoff=15, cell_percentage_cutoff2=0.05, nonz_mean_cutoff=1.12)
ref = ref[:, selected].copy()
vis = vis[:, [g for g in selected if g in vis.var_names]].copy()
print(f"Genes after filtering: {ref.n_vars}")

REF_SIG = f"{PRECOMP_DIR}/level2_c2l_ref_signatures.parquet"
REF_CKPT = f"{CKPT_DIR}/inf_aver_{C2L_MODE}.parquet"
_ref_src = next((p for p in (REF_SIG, REF_CKPT) if os.path.exists(p)), None)
if not TRAIN_C2L:
    if _ref_src is None:
        raise FileNotFoundError(
            f"Precomputed cell2location reference signature not found (looked in {REF_SIG} and "
            f"{REF_CKPT}). Repoint paths for this server (see CLAUDE.md), or set TRAIN_C2L = True "
            "if a GPU is available.")
    inf_aver = pd.read_parquet(_ref_src)   # read the transcript table (parquet)
    print(f"Loaded precomputed reference signature from {_ref_src} (skipped {REF_EPOCHS}-epoch GPU training)")
else:
    RegressionModel.setup_anndata(ref, layer="counts", batch_key="donor_id", labels_key="cell_state_for_c2l")
    ref_model = RegressionModel(ref)
    ref_model.train(max_epochs=REF_EPOCHS, batch_size=10000)
    ref = ref_model.export_posterior(ref, sample_kwargs={"num_samples": 100, "batch_size": 10000})
    inf_aver = ref.varm["q05_per_cluster_mu_fg"]
    inf_aver.to_parquet(REF_CKPT)
    print(f"Saved reference signature checkpoint -> {REF_CKPT}")
print(f"Reference signature: {inf_aver.shape} (genes x cell types)")

**TASK 5.3:** Train the spatial mapping model and export cell-type abundance per spot.

In [ ]:
# [KEEP-IN-STUDENT]
vis = vis[:, [g for g in inf_aver.index if g in vis.var_names]].copy()
inf_aver_aligned = inf_aver.loc[vis.var_names]

MAP_FILE = f"{PRECOMP_DIR}/level2_c2l_AT10_mapped.h5ad"
MAP_CKPT = f"{CKPT_DIR}/vis_AT10_{C2L_MODE}.h5ad"
_map_src = next((p for p in (MAP_FILE, MAP_CKPT) if os.path.exists(p)), None)
if not TRAIN_C2L:
    if _map_src is None:
        raise FileNotFoundError(
            f"Precomputed AT10 mapping not found (looked in {MAP_FILE} and {MAP_CKPT}). Repoint paths "
            "for this server (see CLAUDE.md), or set TRAIN_C2L = True if a GPU is available.")
    vis = sc.read_h5ad(_map_src)   # load the AnnData object from disk
    print(f"Loaded precomputed AT10 mapping from {_map_src} (skipped {MAP_EPOCHS}-epoch GPU training)")
else:
    Cell2location.setup_anndata(vis, layer="counts", batch_key="sample_name" if "sample_name" in vis.obs else None)
    sp_model = Cell2location(vis, cell_state_df=inf_aver_aligned, N_cells_per_location=30, detection_alpha=200)
    # Paper used batch_size ~= 25% of spots per tumour (Methods), not full-batch.
    sp_model.train(max_epochs=MAP_EPOCHS, batch_size=max(int(0.25 * vis.n_obs), 1))
    vis = sp_model.export_posterior(vis, sample_kwargs={"num_samples": 100, "batch_size": vis.n_obs})
    vis.write_h5ad(MAP_CKPT)
    print(f"Saved AT10 mapping checkpoint -> {MAP_CKPT}")

abundance = vis.obsm["q05_cell_abundance_w_sf"] if "q05_cell_abundance_w_sf" in vis.obsm else \
            vis.obs[[c for c in vis.obs.columns if c.startswith("q05")]]
print(f"Cell-type abundance per spot: {abundance.shape}")
print(abundance.describe().T[["mean", "std", "max"]].round(2))

**TASK 5.4:** Plot a few cell-type abundance maps on tissue.

In [ ]:
# --- TASK 5.4: plot a few cell-type abundance maps ---
# WHAT & WHY: see WHERE each cell type sits, now that spots are deconvolved into per-type abundances.
# HOW: top = abundance.mean().nlargest(4).index; for each type scatter vis.obsm['spatial'] coloured by
#      abundance[type] (cmap='viridis'); or sq.pl.spatial_scatter(vis, color=type).
# DOCS: https://squidpy.readthedocs.io/en/stable/api/squidpy.pl.spatial_scatter.html

# ---- provided — read it and run it (mostly plotting/formatting) ----
top_types = abundance.mean().nlargest(4).index.tolist()
fig, axes = plt.subplots(1, len(top_types), figsize=(4.2 * len(top_types), 4.2))   # set up the figure and its panel(s)
for ax, ct in zip(axes, top_types):
    vals = abundance[ct].to_numpy()
    coords = vis.obsm["spatial"]
    sca = ax.scatter(coords[:, 0], coords[:, 1], c=vals, cmap="viridis", s=8)   # scatter each point at its (x, y); colour = the c= values
    ax.invert_yaxis()   # flip the y-axis so it matches the tissue-image orientation
    ax.set_aspect("equal")   # keep x and y on the same scale so the tissue isn't distorted
    ax.set_title(ct)   # give this panel a title
    ax.axis("off")   # hide the axis frame and ticks
    fig.colorbar(sca, ax=ax, shrink=0.7)   # add a colour-scale legend
plt.tight_layout()   # adjust spacing so panels don't overlap
plt.show()   # display the figure

**QUESTION:** Compare these deconvolved abundance maps to the blended Section 4 scores. Are the malignant-state spatial patterns sharper now? Does any region's *dominant* cell type surprise you given what the naive expression clustering (Section 3) suggested was there?

---

## 6. Spatial niches and intermixing

Individual cell-type abundances are noisy spot-by-spot. **NMF** on the abundance matrix finds
recurring *co-occurrence patterns* — niches — the way the original study does.

**TASK 6.1:** Run NMF on the cell2location abundance matrix with a few different factor counts.

In [ ]:
# --- TASK 6.1: NMF on the abundance matrix -> niches ---
# WHAT & WHY: NMF finds recurring cell-type CO-OCCURRENCE patterns (niches) across spots.
# HOW: from sklearn.decomposition import NMF
#      for n in (5, 8, 12): NMF(n_components=n, init='nndsvda', random_state=0, max_iter=500).fit_transform(abundance.clip(lower=0))
#      pick one factor count (N_NICHES); assign vis.obs['niche'] = argmax of that model's loadings (as strings).
# DOCS: https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.NMF.html

# ---- the plotting is provided — complete the steps (single-cell pipeline you know from Level 1) ----
from sklearn.decomposition import NMF

for n_factors in [5, 8, 12]:
    ...   # your code here — complete this step (you did this in Level 1)
    print(f"n_factors={n_factors}: reconstruction error = {nmf.reconstruction_err_:.1f}")
# Carry forward one choice for the rest of the notebook:
N_NICHES = 8
...   # your code here — complete this step (you did this in Level 1)
vis.obs["niche"] = pd.Categorical(niche_loadings.argmax(axis=1).astype(str))
print(vis.obs["niche"].value_counts())

**TASK 6.2:** Plot niches on tissue, and look at which cell types load most strongly onto each niche factor.

In [ ]:
# --- TASK 6.2: plot niches + which cell types load onto each ---
# WHAT & WHY: map the niches in space, and read off each niche's cell-type composition.
# HOW: plot_spatial_categories(vis, 'niche', spatial_key='spatial')   # helper
#      components = pd.DataFrame(nmf.components_, columns=abundance.columns); sns.heatmap(components)
# DOCS: helper plot_spatial_categories -> src/gbmspace_utils/plotting.py

# ---- provided — read it and run it (mostly plotting/formatting) ----
fig, ax = plt.subplots(figsize=(6, 6))   # set up the figure and its panel(s)
plot_spatial_categories(vis, "niche", spatial_key="spatial", ax=ax)   # plot the categorical labels on the tissue
ax.set_title(f"{N_NICHES} NMF-derived niches")   # give this panel a title
plt.show()   # display the figure

components = pd.DataFrame(nmf.components_, columns=abundance.columns,
                           index=[f"niche_{i}" for i in range(N_NICHES)])
fig, ax = plt.subplots(figsize=(10, 6))   # set up the figure and its panel(s)
sns.heatmap(components.div(components.max(axis=1), axis=0), cmap="viridis", ax=ax)   # draw the matrix as a heatmap (colour = value)
ax.set_title("Cell-type loading per niche (row-normalized)")   # give this panel a title
plt.tight_layout()   # adjust spacing so panels don't overlap
plt.show()   # display the figure

**TASK 6.3 — spatial intermixing.** For each spot, compute the Shannon entropy of its cell-type abundance distribution — a high-entropy spot has many cell types evenly mixed; a low-entropy spot is dominated by one.

In [ ]:
# --- TASK 6.3: spatial intermixing (Shannon entropy) ---
# WHAT & WHY: per spot, entropy of its cell-type abundance -> HIGH = many types evenly mixed, LOW = one type dominates.
# HOW: from scipy.stats import entropy
#      props = abundance.clip(lower=0); props = props.div(props.sum(axis=1), axis=0).fillna(0)
#      vis.obs['intermixing_entropy'] = props.apply(lambda r: entropy(r + 1e-12), axis=1); plot it on tissue.
# DOCS: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.entropy.html

# ---- the plotting is provided — complete the steps (single-cell pipeline you know from Level 1) ----
from scipy.stats import entropy

props = abundance.clip(lower=0)
props = props.div(props.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
...   # your code here — complete this step (you did this in Level 1)

fig, ax = plt.subplots(figsize=(6, 6))   # set up the figure and its panel(s)
coords = vis.obsm["spatial"]
sca = ax.scatter(coords[:, 0], coords[:, 1], c=vis.obs["intermixing_entropy"], cmap="magma", s=8)   # scatter each point at its (x, y); colour = the c= values
ax.invert_yaxis()   # flip the y-axis so it matches the tissue-image orientation
ax.set_aspect("equal")   # keep x and y on the same scale so the tissue isn't distorted
ax.set_title("Spatial intermixing (entropy)")   # give this panel a title
ax.axis("off")   # hide the axis frame and ticks
fig.colorbar(sca, ax=ax, shrink=0.7)   # add a colour-scale legend
plt.show()   # display the figure
print(vis.obs.groupby("niche")["intermixing_entropy"].mean().sort_values())   # group the rows to aggregate per group

**QUESTION:** Which niche has the lowest intermixing entropy (most "pure")? Which has the highest? Does low entropy correspond to a niche you'd expect to be compositionally homogeneous (e.g. a dense malignant core) versus a mixed immune/stromal region?

---

## 7. Spatial neighborhood analysis, two ways

**TASK 7.1 — squidpy.** Build the spatial neighbor graph and compute neighborhood enrichment between niches.

In [ ]:
# --- TASK 7.1: squidpy neighborhood enrichment ---
# WHAT & WHY: which niches sit next to which? squidpy builds a spatial graph and tests neighbour enrichment.
# HOW: sq.gr.spatial_neighbors(vis, coord_type='generic', n_neighs=6)
#      sq.gr.nhood_enrichment(vis, cluster_key='niche'); sq.pl.nhood_enrichment(vis, cluster_key='niche')
# DOCS: https://squidpy.readthedocs.io/en/stable/api/squidpy.gr.spatial_neighbors.html  |  https://squidpy.readthedocs.io/en/stable/api/squidpy.gr.nhood_enrichment.html

# ---- the plotting is provided — complete the steps (single-cell pipeline you know from Level 1) ----
...   # your code here — complete this step (you did this in Level 1)
sq.pl.nhood_enrichment(vis, cluster_key="niche", figsize=(6, 5), annotate=False)

**TASK 7.2 — the paper's own method.** Build the same kind of "which niches are near which" picture a different way: pairwise minimum spot-distance via a k-d tree, summarized at the 25th percentile (`gbmspace_utils.spatial_proximity_network` — this mirrors the paper's actual Fig. 2E/3C/6E/7E method, an alternative to squidpy's enrichment z-scores).

In [ ]:
# --- TASK 7.2: the paper's proximity-network method ---
# WHAT & WHY: an alternative to squidpy -- pairwise minimum spot-distance between niches (25th percentile).
# HOW: prox = spatial_proximity_network(vis, cluster_key='niche', spatial_key='spatial', percentile=25)  # helper
#      sns.heatmap(prox, cmap='viridis_r')   # reversed cmap: closer (smaller distance) = brighter
# DOCS: helper spatial_proximity_network -> src/gbmspace_utils/analysis.py

# ---- the plotting is provided — complete the steps (single-cell pipeline you know from Level 1) ----
...   # your code here — complete this step (you did this in Level 1)
fig, ax = plt.subplots(figsize=(7, 6))   # set up the figure and its panel(s)
sns.heatmap(prox, cmap="viridis_r", ax=ax)  # reversed: closer (smaller distance) = brighter
ax.set_title("Niche-niche proximity (25th-pctile nearest distance, smaller=closer)")   # give this panel a title
plt.tight_layout()   # adjust spacing so panels don't overlap
plt.show()   # display the figure

**QUESTION:** Do squidpy's neighborhood-enrichment z-scores and the proximity-distance heatmap agree on which niches are spatial neighbors? Where (if anywhere) do they disagree, and why might two reasonable methods for "spatial closeness" give different answers?

---

## 8. Cell-cell communication with LIANA

The published study runs **LIANA** (a consensus of several ligand-receptor methods) comparing
TME states co-localized with dev-like niches vs. those co-localized with gliosis/hypoxia
niches. We skip their cross-donor Tensor-cell2cell step (not meaningful with only 2 donors) but
reproduce the core comparison directly on our own spots.

**TASK 8.1:** Define two spot groups directly from the per-spot malignant-axis scores
already computed in Section 4 (`state_scores`, on the full normalized `adata` — deliberately
*not* the heavily gene-filtered `vis` from cell2location, so LIANA sees the full gene panel
rather than just the ~15,900 genes that survived cell2location's deconvolution-specific
filter) — "dev-like-dominant" (OPC-like / OPC-NPC-like / OPC-neuronal-like / NPC-neuronal-like
/ AC-progenitor-like) vs. "gliosis/hypoxia-dominant" (AC-gliosis-like / Gliosis-like /
Hypoxic) — leaving Proliferative-dominant spots out of the comparison (neither end of the
trajectory).

> **How LIANA works (plain-language).** Cells communicate through **ligand–receptor** pairs: one
> group secretes a ligand, another expresses the matching receptor. For a given *source group →
> target group*, LIANA scores how "active" each known ligand–receptor pair is — roughly, ligand
> expression in the source combined with receptor expression in the target, plus how **specific**
> that pair is to those groups. Many scoring methods exist (CellPhoneDB, NATMI, Connectome, …), each
> with its own formula; **LIANA runs several and aggregates their rankings into a consensus**
> (`rank_aggregate`), so you don't depend on any single method's quirks.
>
> **Inputs:** (a) a grouping of the observations (here, our two niche groups) and (b) a database of
> known ligand–receptor pairs (`resource_name="consensus"`). **Output** (in `.uns["liana_res"]`) is a
> table of source→target→ligand→receptor rows with a `magnitude_rank` (interaction strength) and a
> `specificity_rank` — **lower rank = stronger / more specific**. Important caveat: this scores
> *co-expression patterns between groups* (potential communication), **not** physical proof that two
> cells actually touched.

In [ ]:
# --- TASK 8.1: define two spot groups from the axis scores ---
# WHAT & WHY: split spots into 'dev-like-dominant' vs 'gliosis-hypoxia-dominant' (drop the rest) for the CCC comparison.
# HOW: DEV_LIKE = ['OPC-like','OPC-NPC-like','OPC-neuronal-like','NPC-neuronal-like','AC-progenitor-like']
#      GLIOSIS_HYPOXIA = ['AC-gliosis-like','Gliosis-like','Hypoxic']
#      dominant_axis = assign_dominant_state(state_scores)   # helper; state_scores from Section 4
#      adata.obs['niche_group'] = np.select([dominant_axis.isin(DEV_LIKE), dominant_axis.isin(GLIOSIS_HYPOXIA)],
#                                            ['dev-like-dominant','gliosis-hypoxia-dominant'], default='other')
# DOCS: helper assign_dominant_state -> src/gbmspace_utils/analysis.py

# ---- the plotting is provided — complete the steps (single-cell pipeline you know from Level 1) ----
DEV_LIKE_STATES = ["OPC-like", "OPC-NPC-like", "OPC-neuronal-like", "NPC-neuronal-like", "AC-progenitor-like"]
GLIOSIS_HYPOXIA_STATES = ["AC-gliosis-like", "Gliosis-like", "Hypoxic"]

...   # your code here — complete this step (you did this in Level 1)

adata.obs["niche_group"] = np.select(
    [dominant_axis.isin(DEV_LIKE_STATES).to_numpy(), dominant_axis.isin(GLIOSIS_HYPOXIA_STATES).to_numpy()],
    ["dev-like-dominant", "gliosis-hypoxia-dominant"],
    default="other",
)
print(adata.obs["niche_group"].value_counts())

**TASK 8.2:** Run LIANA's consensus ligand-receptor scoring with `niche_group` as the
grouping variable — this scores every source-group -> target-group pair, so with two groups you
get all four directions, including the two cross-group ("between niche") directions we actually
care about.

In [ ]:
# --- TASK 8.2: run LIANA consensus ligand-receptor scoring ---
# WHAT & WHY: score cell-cell communication between the two spot groups (consensus of several LR methods).
# HOW: import liana as li
#      vis_ccc = adata[adata.obs['niche_group'] != 'other'].copy(); vis_ccc.obs['niche_group'] = vis_ccc.obs['niche_group'].astype('category')
#      li.mt.rank_aggregate(vis_ccc, groupby='niche_group', resource_name='consensus', use_raw=True)
#      # results land in vis_ccc.uns['liana_res']
# DOCS: https://liana-py.readthedocs.io/en/latest/notebooks/basic_usage.html

# ---- the plotting is provided — complete the steps (single-cell pipeline you know from Level 1) ----
import liana as li

vis_ccc = adata[adata.obs["niche_group"] != "other"].copy()
vis_ccc.obs["niche_group"] = vis_ccc.obs["niche_group"].astype("category")
print(f"Spots used for LIANA: {vis_ccc.n_obs} ({vis_ccc.obs['niche_group'].value_counts().to_dict()})")

...   # your code here — complete this step (you did this in Level 1)
liana_res = vis_ccc.uns["liana_res"]
print(f"\nLIANA interactions scored: {len(liana_res)}")
print(liana_res["source"].astype(str).str.cat(liana_res["target"].astype(str), sep=" -> ").value_counts())

**TASK 8.3:** Look at the top cross-group interactions in each direction (dev-like spots
signalling to gliosis/hypoxia spots, and vice versa), ranked by LIANA's aggregate
`magnitude_rank` (lower = stronger consensus across methods).

In [ ]:
# --- TASK 8.3: top cross-group interactions per direction ---
# WHAT & WHY: read off the strongest signalling in each direction (lower magnitude_rank = stronger consensus).
# HOW: liana_res = vis_ccc.uns['liana_res']
#      for src, tgt in [('dev-like-dominant','gliosis-hypoxia-dominant'), ('gliosis-hypoxia-dominant','dev-like-dominant')]:
#          cross = liana_res[(liana_res['source']==src) & (liana_res['target']==tgt)].sort_values('magnitude_rank')
#          print(cross[['ligand_complex','receptor_complex','magnitude_rank']].head(10))
# DOCS: https://liana-py.readthedocs.io/en/latest/notebooks/basic_usage.html

# ---- provided — read it and run it (mostly plotting/formatting) ----
for src, tgt in [("dev-like-dominant", "gliosis-hypoxia-dominant"), ("gliosis-hypoxia-dominant", "dev-like-dominant")]:
    cross = liana_res[(liana_res["source"] == src) & (liana_res["target"] == tgt)].sort_values("magnitude_rank")
    print(f"\nTop 10 {src} -> {tgt}:")
    print(cross[["ligand_complex", "receptor_complex", "magnitude_rank", "specificity_rank"]].head(10).to_string(index=False))

li.pl.dotplot(adata=vis_ccc, colour="magnitude_rank", size="specificity_rank",
              inverse_size=True, inverse_colour=True,
              source_labels=["dev-like-dominant", "gliosis-hypoxia-dominant"],
              target_labels=["dev-like-dominant", "gliosis-hypoxia-dominant"],
              top_n=15, orderby="magnitude_rank", orderby_ascending=True, figure_size=(9, 7))
plt.show()   # display the figure

**QUESTION:** Which ligand-receptor pairs are specific to one direction (e.g. dev-like
spots signalling to gliosis/hypoxia spots) rather than the reverse? Do any involve genes you
already recognise from the hypoxia/gliosis marker sets in `MALIGNANT_AXIS_MARKERS` (e.g.
`VEGFA`)? What would that suggest about how these two ends of the trajectory interact spatially,
rather than just co-occurring?

---

## 9. Revealing the paper

**de Jong, Memi, Gracia, Lazareva et al. "A spatiotemporal cancer cell trajectory
underlies glioblastoma heterogeneity." bioRxiv 2025.05.13.653495.** Companion website:
[gbmspace.org](https://www.gbmspace.org/). The data you have been working with (AT10 and
AT14, snRNA-seq + Visium) are two of the 12 IDH-wildtype glioblastoma tumours profiled in
this study (1,025,329 nuclei total across the cohort).

**Key findings, for comparison against your own results:**
- Malignant cells occupy a **continuous trajectory**, not discrete subtypes: from
  developmental-like states (OPC-like, NPC-neuronal-like, AC-progenitor-like) through a
  **gliosis** and **hypoxia** axis — what was historically called "mesenchymal-like" (MES1/2)
  in the Neftel et al. 2019 framework, but the authors show classical EMT regulators
  (`SNAI1/2`, `TWIST1/2`, `ZEB1/2`) are *not* specifically enriched there, arguing against an
  EMT interpretation.
- This trajectory maps onto **spatial zonation**: AC-progenitor-like cells dominate near the
  tumour core / infiltrating edge; gliosis and hypoxic states concentrate deep in the tumour,
  around and within necrotic regions — exactly the `AQP4` → `ABCC3` → `AKAP12` → `HILPDA`
  gradient you looked for in Section 4.
- Spatial **niches** were derived the same way you just did it: NMF on cell2location
  cell-state abundances (16 factors per tumour in the paper, clustered into ~14-16 recurrent
  niches across the cohort), cross-validated against pathologist-annotated IvyGAP regions
  (leading edge, infiltrating tumour, cellular tumour, necrosis, pseudopalisading cells,
  perinecrotic zone, microvascular proliferation).
- A major caveat the authors flag explicitly: **single-biopsy sampling can be misleading** —
  one of their tumours (AT10, the same donor in your data!) showed a different dominant
  malignant state in each of its 4 sampled sites, challenging the idea of one fixed
  "subtype" per tumour.

**TASK 9.1:** Now that you know the source, compare your own niche map and axis scores
against the paper's actual cell2location/niche outputs for this exact section (these were
withheld from your input data — load them now from the answer-key file for comparison only).

In [ ]:
# --- TASK 9.1: compare against the withheld answer key ---
# WHAT & WHY: now the paper is revealed -- compare your niches/axis to the paper's actual outputs for THIS section.
# HOW: answer_key = sc.read_h5ad("<AT10 answer-key .h5ad>")
#      the answer blocks are stored in .var['feature_types']; pull the relevant one(s) and correlate against
#      your NMF niche loadings / per-spot axis scores (e.g. pandas .corrwith / numpy corrcoef).
# DOCS: https://anndata.readthedocs.io/en/stable/generated/anndata.read_h5ad.html

# ---- provided — read it and run it (mostly plotting/formatting) ----
answer_key = sc.read_h5ad(DATA_DIR / "answer_keys/AT10-BRA-5-FO-1_2_answer_key.h5ad")   # load the AnnData object from disk
print(answer_key.var["feature_types"].value_counts())

def extract_answer_block(ak, feature_type):
    mask = (ak.var["feature_types"] == feature_type).to_numpy()
    block = ak.X[:, mask]
    block = block.toarray() if hasattr(block, "toarray") else np.asarray(block)
    return pd.DataFrame(block, index=ak.obs_names, columns=ak.var_names[mask])

niche_answer = extract_answer_block(answer_key, "Spatial niche abundances")
state_answer = extract_answer_block(answer_key, "Cell state abundances")
print(f"\nAnswer-key niche columns ({niche_answer.shape[1]}): {list(niche_answer.columns)}")
print(f"Answer-key cell-state columns ({state_answer.shape[1]}): {list(state_answer.columns)}")

common_spots = vis.obs_names.intersection(niche_answer.index)
print(f"\nSpots in common with answer key: {len(common_spots)} / {vis.n_obs}")

niche_onehot = pd.get_dummies(vis.obs.loc[common_spots, "niche"])
joint = np.hstack([niche_onehot.to_numpy(), niche_answer.loc[common_spots].to_numpy()])
full_corr = np.corrcoef(joint, rowvar=False)
corr = pd.DataFrame(full_corr[:niche_onehot.shape[1], niche_onehot.shape[1]:],
                     index=niche_onehot.columns, columns=niche_answer.columns)
print("\nCorrelation: your NMF niche (rows) vs the paper's niche abundance (cols):")
print(corr.round(2))
print("\nBest-matching paper niche per your niche (by |correlation|):")
print(corr.abs().idxmax(axis=1))

fig, ax = plt.subplots(figsize=(8, 6))   # set up the figure and its panel(s)
sns.heatmap(corr, cmap="RdBu_r", center=0, ax=ax)   # draw the matrix as a heatmap (colour = value)
ax.set_title("Your niches vs the paper's niche abundances (correlation)")   # give this panel a title
plt.tight_layout()   # adjust spacing so panels don't overlap
plt.show()   # display the figure

**QUESTION:** Where does your independent analysis agree with the published result, and where does it diverge? For any divergence, what's your best hypothesis — different gene filtering, different epoch budget (FAST vs FULL), different niche factor count, or something about how the reference was built in Level 1?

---

## 10. Secondary check: does the same pattern hold in tumour 2? (AT14)

Everything above used AT10 only. The paper's own caveat from Section 9 — that a single biopsy
can be misleading, and that AT10 itself showed different dominant states across its 4 sampled
sites — is exactly why a second, independent tumour is worth checking. `AT14`'s Visium section
has **no IvyGAP histopathology overlay**, so this is a cross-tumour sanity check against AT10,
not a check against ground truth. It's also intentionally **condensed**: the reference
signature from Level 1 is reused as-is (only a new spatial-mapping model is trained), and we
don't repeat the squidpy-vs-k-d-tree neighborhood comparison or LIANA here — those stay AT10-only.

**TASK 10.1:** Load, QC, normalize, and score the malignant-state axis on AT14's spots, the
same way as Sections 2-4.

In [ ]:
# --- TASK 10.1: repeat sections 2-4 on AT14 ---
# WHAT & WHY: a condensed second-tumour check -- same load / QC / normalize / axis-score steps as AT10.
# HOW: adata14 = sc.read_h5ad("<AT14 Visium .h5ad>")
#      reuse EXACTLY the calls from Sections 2-4: mt flag + sc.pp.calculate_qc_metrics + light filter +
#      normalize_total + log1p + highly_variable_genes + score_axis(adata14, MALIGNANT_AXIS_MARKERS).
# DOCS: https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.calculate_qc_metrics.html  |  https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.normalize_total.html

# ---- the plotting is provided — complete the steps (single-cell pipeline you know from Level 1) ----
VISIUM_AT14 = DATA_DIR / "visium/level2_prepared/AT14-BRA-4-FO-2_1_student.h5ad"
adata14 = sc.read_h5ad(VISIUM_AT14)   # load the AnnData object from disk
adata14.var["mt"] = adata14.var_names.str.startswith("MT-")   # flag mitochondrial genes
...   # your code here — complete this step (you did this in Level 1)

n0 = adata14.n_obs
adata14 = adata14[(adata14.obs["total_counts"] >= 500) & (adata14.obs["n_genes_by_counts"] >= 250)].copy()
...   # your code here — complete this step (you did this in Level 1)
print(f"AT14 spots: {n0} -> {adata14.n_obs}  |  genes: {adata14.n_vars}")

adata14.layers["counts"] = adata14.X.copy()
...   # your code here — complete this step (you did this in Level 1)
adata14.raw = adata14

...   # your code here — complete this step (you did this in Level 1)
adata14.obs["malignant_class"] = dominant14.map(MAJOR_CLASS_OF)
print("\nAT14 dominant malignant-class fractions:")
print(adata14.obs["malignant_class"].value_counts(normalize=True).round(3))

**TASK 10.2:** Map cell2location onto AT14 — reusing the **same** reference signature
(`inf_aver`) fit once in Section 5, training only a new spatial-mapping model — then derive
niches the same way as Section 6.

In [ ]:
# [KEEP-IN-STUDENT]
shared14 = sorted(set(inf_aver.index) & set(adata14.var_names))
vis14 = adata14.copy()[:, shared14].copy()
inf_aver_aligned14 = inf_aver.loc[shared14]
print(f"AT14 genes shared with the Level 1 reference signature: {len(shared14)}")

MAP_FILE14 = f"{PRECOMP_DIR}/level2_c2l_AT14_mapped.h5ad"
MAP_CKPT14 = f"{CKPT_DIR}/vis_AT14_{C2L_MODE}.h5ad"
_map_src14 = next((p for p in (MAP_FILE14, MAP_CKPT14) if os.path.exists(p)), None)
if not TRAIN_C2L:
    if _map_src14 is None:
        raise FileNotFoundError(
            f"Precomputed AT14 mapping not found (looked in {MAP_FILE14} and {MAP_CKPT14}). Repoint "
            "paths for this server (see CLAUDE.md), or set TRAIN_C2L = True if a GPU is available.")
    vis14 = sc.read_h5ad(_map_src14)   # load the AnnData object from disk
    print(f"Loaded precomputed AT14 mapping from {_map_src14} (skipped {MAP_EPOCHS}-epoch GPU training)")
else:
    Cell2location.setup_anndata(vis14, layer="counts", batch_key="sample_name" if "sample_name" in vis14.obs else None)
    sp_model14 = Cell2location(vis14, cell_state_df=inf_aver_aligned14, N_cells_per_location=30, detection_alpha=200)
    _bs14 = max(int(0.25 * vis14.n_obs), 2)
    if vis14.n_obs % _bs14 == 1:  # last mini-batch of 1 sample causes 0-d tensor error in PyTorch
        _bs14 += 1
    sp_model14.train(max_epochs=MAP_EPOCHS, batch_size=_bs14)
    vis14 = sp_model14.export_posterior(vis14, sample_kwargs={"num_samples": 100, "batch_size": vis14.n_obs})
    vis14.write_h5ad(MAP_CKPT14)
    print(f"Saved AT14 mapping checkpoint -> {MAP_CKPT14}")

abundance14 = vis14.obsm["q05_cell_abundance_w_sf"] if "q05_cell_abundance_w_sf" in vis14.obsm else \
              vis14.obs[[c for c in vis14.obs.columns if c.startswith("q05")]]
print(f"AT14 cell-type abundance per spot: {abundance14.shape}")

nmf14 = NMF(n_components=N_NICHES, init="nndsvda", random_state=0, max_iter=500)
niche_loadings14 = nmf14.fit_transform(abundance14.clip(lower=0))
vis14.obs["niche"] = pd.Categorical(niche_loadings14.argmax(axis=1).astype(str))

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))   # set up the figure and its panel(s)
plot_spatial_categories(vis14, "niche", spatial_key="spatial", ax=axes[0])   # plot the categorical labels on the tissue
axes[0].set_title(f"AT14: {N_NICHES} NMF-derived niches")   # give this panel a title
plt.tight_layout()   # adjust spacing so panels don't overlap
plt.show()   # display the figure
print(vis14.obs["niche"].value_counts())

**TASK 10.3:** Compare AT10 vs AT14 directly — does the dev-like -> gliosis -> hypoxia
axis, and a comparable niche structure, show up in both tumours?

In [ ]:
# --- TASK 10.3: compare AT10 vs AT14 composition ---
# WHAT & WHY: does the dev-like -> gliosis -> hypoxia axis show up in both tumours, or does one skew?
# HOW: adata.obs['malignant_class'] = dominant_axis.map(MAJOR_CLASS_OF)   # same for adata14
#      pd.DataFrame({'AT10': adata.obs['malignant_class'].value_counts(normalize=True),
#                    'AT14': adata14.obs['malignant_class'].value_counts(normalize=True)}).fillna(0)
# DOCS: helper MAJOR_CLASS_OF -> src/gbmspace_utils/analysis.py

# ---- provided — read it and run it (mostly plotting/formatting) ----
adata.obs["malignant_class"] = dominant_axis.map(MAJOR_CLASS_OF)  # dominant_axis from Section 8

cross_tumor = pd.DataFrame({
    "AT10": adata.obs["malignant_class"].value_counts(normalize=True),
    "AT14": adata14.obs["malignant_class"].value_counts(normalize=True),
}).fillna(0).round(3)
print("Dominant malignant-class fraction per spot, AT10 vs AT14:")
print(cross_tumor)

fig, ax = plt.subplots(figsize=(7, 4))   # set up the figure and its panel(s)
cross_tumor.plot(kind="bar", ax=ax)
ax.set(ylabel="fraction of spots", title="Malignant-class composition: AT10 vs AT14")
plt.tight_layout()   # adjust spacing so panels don't overlap
plt.show()   # display the figure

print(f"\nAT10 niches: {N_NICHES} (from {vis.n_obs} spots) | AT14 niches: {N_NICHES} (from {vis14.n_obs} spots)")
print("Note: AT14 has no IvyGAP/cell2location answer key, so this is a cross-tumour check, "
      "not a check against ground truth (TASK 9.1's answer-key comparison stays AT10-only).")

**QUESTION:** Is the malignant-class composition similar between AT10 and AT14, or does
one tumour skew much more dev-like / gliosis-hypoxia than the other? Given the paper's own
caveat about single-biopsy sampling (Section 9), how confident should you be that either
section alone represents "the" malignant-state distribution of its tumour?

---

## 11. Write-up

**TASK 11.1:** Reproduce one specific published figure panel using your own pipeline, and
write a short paragraph comparing your result to the original. We reproduce the spatial
malignant-class map (the panel underlying the paper's Fig. 2/3 zonation claim) directly from
our own AT10 pipeline.

In [ ]:
# --- TASK 11.1: reproduce the spatial malignant-class map ---
# WHAT & WHY: recreate the panel behind the paper's Fig. 2/3 zonation claim from your own AT10 result.
# HOW: scatter adata.obsm['spatial'][:,0/1] coloured by adata.obs['malignant_class'] (map classes -> colours
#      with a seaborn palette); add a legend. Then write your 2-3 sentence comparison in TASK 11.1 below.
# DOCS: https://squidpy.readthedocs.io/en/stable/api/squidpy.pl.spatial_scatter.html   (color='malignant_class' is an easy alternative)

# ---- provided — read it and run it (mostly plotting/formatting) ----
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))   # set up the figure and its panel(s)
classes = sorted(adata.obs["malignant_class"].dropna().unique())
palette = dict(zip(classes, sns.color_palette("Set2", len(classes))))
coords = adata.obsm["spatial"]
colors = adata.obs["malignant_class"].map(palette)
axes[0].scatter(coords[:, 0], coords[:, 1], c=colors, s=8)   # scatter each point at its (x, y); colour = the c= values
axes[0].invert_yaxis()   # flip the y-axis so it matches the tissue-image orientation
axes[0].set_aspect("equal")   # keep x and y on the same scale so the tissue isn't distorted
axes[0].axis("off")   # hide the axis frame and ticks
axes[0].set_title("Our spatial malignant-class map (AT10)")   # give this panel a title
handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=palette[c], label=c, markersize=8)   # add / build the legend
           for c in classes]
axes[0].legend(handles=handles, fontsize=7, loc="lower left")   # add / build the legend

plot_spatial_categories(vis, "niche", spatial_key="spatial", ax=axes[1])   # plot the categorical labels on the tissue
axes[1].set_title(f"Our {N_NICHES}-niche map (AT10), for comparison")   # give this panel a title
plt.tight_layout()   # adjust spacing so panels don't overlap
fig.savefig(NOTEBOOK_DIR / "level2_summary_figure.png",   # save the figure to a file
            dpi=200, bbox_inches="tight")
plt.show()   # display the figure

**TASK 11.1 — Write-up.** In 2-3 sentences, compare your spatial malignant-class map and the niches you recovered to the paper's zonation claim. Back it with your own numbers: the TASK 9.1 niche-correlation values (your NMF niches vs the answer key) and the TASK 10.3 AT10-vs-AT14 dominant-class fractions.


---

## Summary

You have:
1. ✅ QC'd and explored real Visium spatial data
2. ✅ Built a naive (pre-deconvolution) spatial domain map
3. ✅ Seen the malignant cell-state axis directly in space
4. ✅ Mapped your Level 1 reference onto tissue with **cell2location**
5. ✅ Identified spatial niches via NMF, and quantified spatial intermixing
6. ✅ Compared two methods for spatial neighborhood/proximity analysis
7. ✅ Run LIANA cell-cell communication scoring between dev-like and gliosis/hypoxia spots
8. ✅ Compared your independent results against the published findings
9. ✅ Checked whether the same axis/niche pattern holds in a second tumour (AT14)

**Further reading, not built here:** the paper also describes **spaceTree** (joint cell-type and genetic-clone mapping) and **cell2fate** (RNA-velocity-based temporal ordering of malignant
states) — both require data/pipelines beyond this course's current scope (paired snATAC-seq
clone-calling, and spliced/unspliced counts, respectively).